In [ ]:
# FitnessandNutritionsTrack.py

# Importing needed libaries for the codes to work properly

import matplotlib.pyplot as plot
import csv as cs
import xml.etree.ElementTree as xml
from datetime import datetime as dt


#Categories and Meals

WORKOUT_CATEGORIES = ["Cardio", "Strength","Aerobic","Flexibility"]
MEAL_CATEGORIES = ["Breakfast","Bruch", "Lunch", "Dinner", "Snack"]

# Defining profile useraccountname to profile information and the list of entries.

# Every profile is a dictionary with key values: "0profile" and "entries"

profiles = {}

# PROFILE MANAGEMENT Section


def create_loaded_profile(useraccountname):
    """ Loading existing profile or creating a new profile if the account does not exist """

    if useraccountname not in profiles:
        # Initializing profile with default None values for stats.
        profiles[useraccountname] = {
            'profile': {
                'weight': None,
                'height': None,
                'fitness_goal': None
            },
            'entries': []  # Listing all entries (everyday entry is a dictionary by itself)
        }

        print(f"\n -> Profile does not exist... Creating new profile account for the user: {useraccountname}")
    else:
        print(f"\n -> Profile loaded for user successfuly: {useraccountname}")

    return useraccountname


def change_profile(useraccountname):

    create_loaded_profile(useraccountname)

def view_profile(useraccountname):

    """Allowing the user to view his profile account"""
    print(
        f"Your current profile account stats:\nWeight: {profiles[useraccountname]['profile']['weight']}\nHeight: {profiles[useraccountname]['profile']['height']}\nGoals: {profiles[useraccountname]['profile']['fitness_goal']}\n"
    )


def edit_profile(useraccountname):
    """Allowing the user to edit his profile account"""

    profile_data = profiles[useraccountname]['profile']

    print(
        f"Your current profile account stats:\nWeight: {profiles[useraccountname]['profile']['weight']}\nHeight: {profiles[useraccountname]['profile']['height']}\nGoals: {profiles[useraccountname]['profile']['fitness_goal']}\n"
    )

    weight = input("Enter new weight: ")
    height = input("Enter new height: ")
    fitness_goal = input("Enter new fitness goal: ")

    # Updating your profile account if input is provided.
    if weight:
        profile_data['weight'] = weight
    if height:
        profile_data['height'] = height
    if fitness_goal:
        profile_data['fitness_goal'] = fitness_goal

    print("Profile account was  updated.")


# ENTRY MANAGEMENT Section

def add_entry(useraccountname):
    """Allowing the user to add a new entry"""

    entry_type = input("Enter type ('Workout' or 'Meal'): ").strip().title()

    if entry_type not in ["Workout", "Meal"]:
        print('Invalid type. Entry must be "Workout" or "Meal".')
        return


    date_str = input("Enter date and use this format (YYYY-MM-DD): ").strip()
    try:
        entry_date = dt.strptime(date_str, "%Y-%m-%d").date()
    except ValueError:
        print("Invalid date format. Use YYYY-MM-DD.")
        return


    if entry_type == "Workout":
        print("Choose one of the predeffined workout categories: ", WORKOUT_CATEGORIES)
        category = input(" choose and tyoe one of the predeffined category or type a new one: ").strip().title()
        duration_quantity = input("Enter duration for workout: ").strip()

    else:
        print("Predefined meal categories: ", MEAL_CATEGORIES)
        category = input("choose and tyoe category or type a new one: ").strip().title()
        duration_quantity = input("Enter quantity for meal: ").strip()

    try:
        calories = int(
            input("Enter calories (burned for workout or consumed for meal): "
                ).strip())
    except ValueError:
        print("Invalid calories value. Enter a value.")
        return


    entry = {
        'type': entry_type,
        'category': category,
        'duration_quantity': duration_quantity,
        'calories': calories,
        'date': entry_date
    }

    profiles[useraccountname]['entries'].append(entry)
    print(f"\n -> {entry_type} entry was added.")

def view_entries(useraccountname):
    entries = profiles[useraccountname]['entries']
    if not entries:
        print("No entries recorded.")
    else:
        i = 1
        for entry in entries:
            print(f"{i}.  [{entry['date']}] \t{entry['type']}\t - {entry['category']}, \t\tDuration/Quantity: {entry['duration_quantity']},\t Calories: {entry['calories']}")
            i += 1


def import_csv_entries(useraccountname):
        """Importing entries from a CSV file."""

        file_path = input("Enter CSV file path: ").strip()
        try:
            with open(file_path, newline='', encoding='utf-8') as csvfile:
                reader = cs.DictReader(csvfile)
                count = 0
                for row in reader:
                    try:
                        entry_type = row['type'].strip().title()
                        category = row['category'].strip().title()
                        duration_quantity = row['duration_quantity'].strip()
                        calories = int(row['calories'])
                        entry_date = dt.strptime(row['date'].strip(), "%Y-%m-%d").date()
                        entry = {
                            'type': entry_type,
                            'category': category,
                            'duration_quantity': duration_quantity,
                            'calories': calories,
                            'date': entry_date
                        }
                        profiles[useraccountname]['entries'].append(entry)
                        count += 1
                    except Exception as e:
                        print("Error processing row:", row, "\n", e)
                print(f"Imported {count} entries from CSV.")
        except FileNotFoundError:
            print("CSV file not found.")

def import_xml_entries(useraccountname):
    """Importing entries from an XML file."""
    print("\n Import XML Entries ")
    file_path = input("Enter the XML file path: ").strip()
    try:
        tree = xml.parse(file_path)
        root = tree.getroot()
        count = 0
        for entry_elem in root.findall('entry'):
            try:
                entry_type = entry_elem.find('type').text.strip().title()
                category = entry_elem.find('category').text.strip().title()
                duration_quantity = entry_elem.find('duration_quantity').text.strip()
                calories = int(entry_elem.find('calories').text.strip())
                date_str = entry_elem.find('date').text.strip()
                entry_date = dt.strptime(date_str, "%Y-%m-%d").date()
                entry = {
                    'type': entry_type,
                    'category': category,
                    'duration_quantity': duration_quantity,
                    'calories': calories,
                    'date': entry_date
                }
                profiles[useraccountname]['entries'].append(entry)
                count += 1
            except Exception as e:
                print("Error processing an XML entry:", e)
        print(f"Imported {count} entries from XML.")
    except FileNotFoundError:
        print("XML file not found.")
    except xml.ParseError:
        print("Error parsing XML file.")



# VISUALIZATION Section

def visualizing_meal_distribution(useraccountname):
    """Visualize distribution of meal categories for a specified month."""
    print("\n--- Visualize Meal Distribution ---")
    year = input("Enter year (YYYY): ").strip()
    month = input("Enter month (MM): ").strip()
    try:
        year = int(year)
        month = int(month)
    except ValueError:
        print("Invalid year or month.")
        return

    from collections import defaultdict
    category_counts = defaultdict(int)
    for entry in profiles[useraccountname]['entries']:
        if entry['type'] == "Meal" and entry['date'].year == year and entry['date'].month == month:
            category_counts[entry['category']] += 1

    if not category_counts:
        print("No meal entries found for the specified month.")
        return

    categories = list(category_counts.keys())
    counts = list(category_counts.values())

    plot.figure(figsize=(10, 8))
    plot.bar(categories, counts, color='red')
    plot.title(f"Meal Category Distribution for {year}-{month:02d}")
    plot.xlabel("Meal Category")
    plot.ylabel("Count")
    plot.show()
    plot.pause(3)
    plot.close()





def visualizing_calories_comparison(useraccountname):
    """Visualizing calories consumed vs. calories burned out on a daily basis."""
    print("\n--- Visualizing Calories Comparison ---")
    daily_stats = {}
    for entry in profiles[useraccountname]['entries']:
        entry_date = entry['date']
        if entry_date not in daily_stats:
            daily_stats[entry_date] = {'consumed': 0, 'burned': 0}
        if entry['type'] == "Meal":
            daily_stats[entry_date]['consumed'] += entry['calories']
        elif entry['type'] == "Workout":
            daily_stats[entry_date]['burned'] += entry['calories']

    if not daily_stats:
        print("No entries to visualize.")
        return

    sorted_dates = sorted(daily_stats.keys())
    dates_str = [d.strftime("%Y-%m-%d") for d in sorted_dates]
    consumed = [daily_stats[d]['consumed'] for d in sorted_dates]
    burned = [daily_stats[d]['burned'] for d in sorted_dates]

    x = range(len(sorted_dates))
    width = 0.40

    plot.figure(figsize=(12, 8))
    plot.bar(x, consumed, width, label='Calories Consumed', color='blue')
    plot.bar([i + width for i in x], burned, width, label='Calories Burned', color='black')
    plot.xlabel("Date")
    plot.ylabel("Calories")
    plot.title("Daily Calories: Consumed vs. Burned")
    plot.xticks([i + width/2 for i in x], dates_str, rotation=90)
    plot.legend()
    plot.tight_layout()
    plot.show()
    plot.pause(3)
    plot.close()

    # Main driver class

# Import necessary modules


#import FitnessandNutritionsTrack as ft


# Interface Section


def main():
    print(" \n\n\n\t\t Hello, Welcome to Fitness and Nutrition Tracker! \n\n\n")
    useraccountname = input("Enter your profile account name: ")
    create_loaded_profile(useraccountname)
    while True:
        print("\n\n  Fitness and Nutrition Track Main Menu")
        print("1. Change profiles")
        print("2. View your profile")
        print("3. Edit your profile")

        print("4. View all entries")
        print("5. Add a new log entry interactively")
        print("6. Import entries from CSV")
        print("7. Import entries from XML")
        print("8. Visualize Meal Distribution")
        print("9. Visualize Calories Consume vs. Calories Burned")
        print("0. Exiting")

        print("\n\n")
        choice = input("Select an option: ")
        print("\n\n")
        if choice == "1":
            useraccountname = input(" Enter profile account name: ")
            change_profile(useraccountname)
        elif choice == "2":
            view_profile(useraccountname)
        elif choice == "3":
            edit_profile(useraccountname)
        elif choice == "4":
            view_entries(useraccountname)
        elif choice == "5":
            add_entry(useraccountname)
        elif choice == "6":
            import_csv_entries(useraccountname)
        elif choice == "7":
            import_xml_entries(useraccountname)
        elif choice == "8":
            visualizing_meal_distribution(useraccountname)
        elif choice == "9":
            visualizing_calories_comparison(useraccountname)
        elif choice == "0":
            print("Exiting...")
            break
        else:
            print("Invalid option. Please select a valid option.")


if __name__ == "__main__":
    main()


